# pyaegean Stage D — train the full model (tags + trees + lemmas)

Trains the complete joint model: GreBerta + tagging heads + biaffine parser + the
edit-script **lemma head**, leakage-clean, then measures every WP3 metric on the UD
test folds from this one checkpoint.

**Target (UD Perseus test): lemma ≥ 87.6** — UAS/LAS/UPOS/UFeats should hold at their
Stage C levels (same heads, same data, one extra loss term).

Run cells top to bottom on a GPU runtime. Cell 5 is an optional re-train; cell 6 needs
one value filled in from cell 4's output — read the notes.

In [ ]:
# 1 — confirm the GPU
import torch
print(torch.cuda.get_device_name(0))
print("bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# 2 — install the package + a fresh clone
!pip install -q "git+https://github.com/ryanpavlicek/pyaegean" torch transformers numpy
!rm -rf pyaegean_repo
!git clone https://github.com/ryanpavlicek/pyaegean.git pyaegean_repo

In [ ]:
# 3 — build the full dataset (sanity: train_sentences 31112, n_scripts ~9263,
#     script_coverage_train ~0.985)
!python pyaegean_repo/training/build_full_dataset.py

In [ ]:
# 4 — train (defaults: 6 epochs, lr 3e-5, batch 32, bf16 auto)
#     each epoch prints: dev UAS / LAS / UPOS and lemma(best=<mode>) <accuracy>
!python pyaegean_repo/training/train_full.py --model bowphs/GreBerta

**5 — decision point.** If the last epoch was still the best on `(LAS + lemma)`,
uncomment and run cell 5. Otherwise skip it.

**Also note the `lemma(best=<mode>)` from the best epoch** — cell 6 uses that mode
(one of `neural-only`, `lookup-first`, `neural-first`, `unseen-neural`).

In [ ]:
# 5 — OPTIONAL longer re-train: uncomment ONLY if still improving at the last epoch
# !python pyaegean_repo/training/train_full.py --model bowphs/GreBerta --epochs 9

In [ ]:
# 6 — headline measurements: set COMPOSE to the best dev mode from cell 4, then run
COMPOSE = "neural-first"  # <-- replace with cell 4's lemma(best=...) mode
!python pyaegean_repo/training/eval_full_ud.py \
    --checkpoint pyaegean_repo/training/out/full/model --compose {COMPOSE} \
    --treebank perseus --split test --out pyaegean_repo/training/out/full/ud-perseus-test.json
!python pyaegean_repo/training/eval_full_ud.py \
    --checkpoint pyaegean_repo/training/out/full/model --compose {COMPOSE} \
    --treebank proiel --split test --out pyaegean_repo/training/out/full/ud-proiel-test.json

In [ ]:
# 7 — save everything to Drive BEFORE the runtime recycles
#     (this checkpoint supersedes Stage C's — it is THE Stage E export artifact)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/pyaegean-stage-d"
!cp -r pyaegean_repo/training/out/full "/content/drive/MyDrive/pyaegean-stage-d/"
!ls -la "/content/drive/MyDrive/pyaegean-stage-d/full" "/content/drive/MyDrive/pyaegean-stage-d/full/model"

## What to bring back

- `full/metrics.json` — training history (incl. the per-mode lemma compositions)
- `full/ud-perseus-test.json` — **the verdict**: lemma ≥ 87.6 completes the WP3
  definition of done on the Perseus fold (POS, morph, lemma, UAS, LAS all above the
  published state of the art, leakage-clean)
- `full/ud-proiel-test.json` — the out-of-domain read

The checkpoint (`full/model/`, ~550 MB: weights + tokenizer + labels + lemma
scripts/lookup) stays in Drive — Stage E exports exactly this to ONNX.